# 02 — Memory Paging & Fault Latency Test



## Objective

Simulate OVRAM (oversubscribed VRAM) workloads. Verify GPU-driven fault
handling and measure page-in latency vs. CUDA Unified Memory baseline.


In [ ]:
# --- Kaggle / fresh-runtime setup ---
import os, shutil, sys, subprocess
from pathlib import Path

SETUP_LOG = Path('/kaggle/working/nexusrt_setup.log')


def _is_repo_root(root: Path) -> bool:
    return (root / 'packaging' / 'pyproject.toml').exists() and (root / 'python' / 'nexusrt').exists()


def _find_repo_root() -> Path:
    input_candidates = [Path('/kaggle/input/nexusrt'), Path('/kaggle/input/nexusrt/nexusrt')]
    if Path('/kaggle/input').exists():
        input_candidates.extend(p.parent.parent for p in Path('/kaggle/input').glob('**/packaging/pyproject.toml'))
    for root in input_candidates:
        if _is_repo_root(root):
            return root

    working_candidates = [Path.cwd(), Path('/kaggle/working/nexusrt')]
    if Path('/kaggle/working').exists():
        working_candidates.extend(p.parent.parent for p in Path('/kaggle/working').glob('**/packaging/pyproject.toml'))
    for root in working_candidates:
        if _is_repo_root(root):
            return root
    raise RuntimeError('NexusRT repo not found. Attach the private Kaggle dataset rbrtsl/nexusrt first.')


def _tail(text: str, n: int = 120) -> str:
    lines = text.splitlines()
    return '\n'.join(lines[-n:])


def _run(cmd, *, check=True, env=None):
    cmd = [str(x) for x in cmd]
    header = '+ ' + ' '.join(cmd)
    print(header)
    with SETUP_LOG.open('a') as f:
        f.write('\n' + header + '\n')
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    if proc.stdout:
        with SETUP_LOG.open('a') as f:
            f.write(proc.stdout)
        print(_tail(proc.stdout, 80))
    if check and proc.returncode:
        print(f'Command failed with exit code {proc.returncode}. Full log: {SETUP_LOG}')
        if proc.stdout:
            print('--- last setup log lines ---')
            print(_tail(proc.stdout, 160))
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout)
    return proc


SETUP_LOG.write_text('NexusRT Kaggle setup log\n')
print('NexusRT Kaggle setup: direct CUDA runtime refresh v4')
src_repo = _find_repo_root().resolve()
work_repo = Path('/kaggle/working/nexusrt')
if str(src_repo).startswith('/kaggle/input/'):
    if work_repo.exists():
        shutil.rmtree(work_repo)
    shutil.copytree(src_repo, work_repo, ignore=shutil.ignore_patterns('.venv', 'packaging/build', '__pycache__', '.pytest_cache'))
    repo = work_repo
else:
    repo = src_repo

os.chdir(repo)
print(f'NexusRT repo: {repo}')
_run(['nvidia-smi'], check=False)
_run(['nvcc', '--version'], check=False)

# Keep source imports available even if editable install fails.
python_dir = str(repo / 'python')
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)
os.environ['PYTHONPATH'] = python_dir + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ.setdefault('NEXUSRT_BOOT_TRACE', '1')

# Kaggle validation uses direct source imports plus the locally built C++ core.
# This avoids editable-install/build-isolation failures in private dataset runs.
print('Using source import + direct CUDA CMake build; skipping editable pip install.')

# Build the CUDA shared library directly so nexusrt._abi can load it.
build_dir = repo / 'packaging' / 'build'
configure_cmd = [
    'cmake', '-S', 'packaging', '-B', str(build_dir), '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    '-DNEXUSRT_ENABLE_CUDA=ON',
    '-DNEXUSRT_ENABLE_METAL=OFF',
    '-DNEXUSRT_BUILD_TESTS=OFF',
    '-DCMAKE_CUDA_ARCHITECTURES=60;75;80;90',
]
configure_ok = _run(configure_cmd, check=False).returncode == 0
if not configure_ok:
    raise RuntimeError(f'CUDA CMake configure failed. These notebooks require CUDA; see {SETUP_LOG}')

build_ok = _run(['cmake', '--build', str(build_dir), '--parallel', '2'], check=False).returncode == 0
if not build_ok:
    raise RuntimeError(f'CUDA CMake build failed. These notebooks require CUDA; see {SETUP_LOG}')

lib_candidates = [
    build_dir / 'outputs' / 'lib' / 'libnexusrt.so',
    build_dir / 'outputs' / 'lib' / 'libnexusrt.dylib',
]
for lib in lib_candidates:
    if lib.exists():
        os.environ['NEXUSRT_LIB'] = str(lib)
        print(f'NEXUSRT_LIB={lib}')
        break
else:
    raise RuntimeError(f'Built NexusRT, but libnexusrt was not found under {build_dir}/outputs/lib. See {SETUP_LOG}')
import nexusrt
info = nexusrt.firmware.init('auto')
print(f'Vendor      : {info.vendor}')
print(f'Arch        : {info.arch}')
print(f'Name        : {info.name}')
print(f'HBM (GB)    : {info.hbm_bytes >> 30}')
print(f'SMs         : {info.sm_count}')
print(f'Features    : {info.features}')



In [ ]:

# Allocate a workload larger than HBM. The fault handler should page in
# blocks on demand from host pinned memory.
import nexusrt.memory as mem

HBM_GB = info.hbm_bytes >> 30
print(f"HBM capacity: {HBM_GB} GB")
target_gb = min(HBM_GB * 2, 160)   # 2× oversubscription, cap at 160GB
print(f"Target workload: {target_gb} GB ({target_gb/HBM_GB:.1f}x HBM)")

# Allocate the workload as many small pages.
PAGE_MB = 16
pages = []
n_pages = (target_gb << 30) // (PAGE_MB << 20)
print(f"Allocating {n_pages} pages of {PAGE_MB} MB each...")
for i in range(min(n_pages, 4096)):  # cap at 4096 for safety
    try:
        a = mem.alloc(PAGE_MB << 20, read_mostly=False)
        pages.append(a)
    except Exception as e:
        print(f"OOM after {len(pages)} pages — {e}")
        break
print(f"live pages: {len(pages)}")


In [ ]:
# Measure page-in latency — touch each page in random order and time it.
import random
random.seed(0)
import time
# Random access pattern → worst-case for any prefetcher.
indices = list(range(len(pages)))
random.shuffle(indices)

sample_n = min(len(indices), 500)
if sample_n == 0:
    raise RuntimeError('No pages allocated; cannot measure page-in latency')

t0 = time.perf_counter()
for i in indices[:sample_n]:
    mem.prefetch(pages[i].ptr, pages[i].bytes)
t1 = time.perf_counter()
nexusrt_page_in_us = (t1 - t0) * 1e6 / sample_n
print(f'NexusRT avg page-in latency: {nexusrt_page_in_us:.1f} us / 16MB page')


In [ ]:
# Baseline: CUDA Unified Memory
# (Only when torch is available — we use it only for the baseline, never
# in NexusRT src/.)
try:
    import torch
    n_int32 = int((target_gb << 30) // 4)
    baseline_buf = torch.empty(n_int32, dtype=torch.int32,
                               device='cuda', pin_memory=False)
    t0 = time.perf_counter()
    for i in range(sample_n):
        _ = baseline_buf[random.randint(0, len(baseline_buf)-1)].item()
    t1 = time.perf_counter()
    cuda_um_us = (t1 - t0) * 1e6 / sample_n
    print(f'CUDA UM avg page-in latency: {cuda_um_us:.1f} us / element')
    print(f'Speedup: {cuda_um_us / nexusrt_page_in_us:.2f}x')
except Exception as e:
    print(f'Baseline skipped: {e}')


In [ ]:

for a in pages: mem.free(a)
nexusrt.firmware.shutdown()
print("✓ Memory paging test complete.")
